# CVAE evaluation

Configure root.

In [ ]:
import sys, subprocess, os
import numpy as np
import pandas as pd
import torch 
import torch.nn as nn
import torch.nn.functional as F
import importlib
%matplotlib inline
from pathlib import Path

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/singlecell-autoencoder.git"
repo_dir = Path("singlecell-autoencoder")
if COLAB:
    root = Path("/content/singlecell-autoencoder")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

# Use GPU if available
generator = torch.Generator().manual_seed(111)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Data path.

In [ ]:
# get model**

if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    data_dir = Path("/content/drive/MyDrive/phenotype-prediction-data")
    sc_path = str(data_dir / "single_cell")

else:
    sc_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/single_cell"

In [ ]:
import src.sc_data; importlib.reload(src.sc_data)
from src.sc_data import get_sc_data, df_to_tensors
import seaborn as sns

data = get_sc_data(path = sc_path)

In [ ]:
from sklearn.model_selection import train_test_split

# Train-test split
test_mask = (data["dose"] == 2) & (data["timepoint"] == 2)
train_df = data[~test_mask]
test_df = data[test_mask]

# Train-val split
train_df, val_df = train_test_split(
    train_df,
    test_size = 0.2,
    random_state = 111,
    shuffle = True
)

# Calculate proportions
num_data = data.shape[0]
train_prop = int(round(train_df.shape[0] * 100 / num_data, 0))
val_prop = int(round(val_df.shape[0] * 100 / num_data, 0))
test_prop = int(round(test_df.shape[0] * 100 / num_data, 0))

print(f"Train:val:test = {train_prop}:{val_prop}:{test_prop}")

Load model.

Sample 1000 cells from test conditions, then compare*